# Masterclass: State Estimation & Filtering for Autonomous Vehicles
## Least Squares, Linear Kalman Filter (LKF), EKF with Analytical Jacobians & The Unscented Kalman Filter (UKF)

Welcome to this comprehensive hands-on educational laboratory on **State Estimation and Filtering in Self-Driving Cars**.

---

### 🗺️ Master Curriculum Roadmap
1. **Module 1: Least Squares Foundations & The Observation Matrix $\mathbf{H}$**
   * Where $\mathbf{H}$ and $\mathbf{P}$ come from: Derivation from the Gauss-Markov BLUE Theorem.
   * Hands-on Calibrations: Ohm's Law Resistance, Wheel Odometry ($r_{\text{eff}}$), and LiDAR 3D Road Plane Fitting.
2. **Module 2: The Discrete Linear Kalman Filter (LKF)**
   * Mathematical Anatomy: $\mathbf{F}, \mathbf{G}, \mathbf{H}, \mathbf{Q}, \mathbf{R}, \mathbf{P}, \mathbf{K}$.
   * 2D Vehicle Tracking with Constant Velocity (CV) Kinematics & $3\sigma$ Uncertainty Envelopes.
3. **Module 3: The Extended Kalman Filter (EKF) & Analytical Jacobians**
   * Formulating and evaluating the 4 Jacobians: $\mathbf{F}_{k-1}, \mathbf{L}_{k-1}, \mathbf{H}_k, \mathbf{M}_k$.
   * Optical Landmark Bearing Estimation (solving `The Nonlinear Kalman Filter/Excersice.ipynb`).
   * 2D Polar Radar Target Tracking (Range $r$ and Azimuth $\phi$).
4. **Module 4: The Unscented Kalman Filter (UKF) & Sigma Points**
   * The Scaled Unscented Transform (UT) with $2n+1$ deterministic sigma points (Zero Jacobians!).
   * Side-by-side Benchmark & Performance Comparison: EKF vs. UKF on severe non-linearities.


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('src'))

import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from position_class import (
    BatchLeastSquares,
    LinearKalmanFilter,
    ExtendedKalmanFilter,
    UnscentedKalmanFilter,
    LandmarkBearingEKF,
    LandmarkBearingUKF,
    Radar2DTargetTrackerEKF,
    create_2d_constant_velocity_tracker,
    ohms_law_example,
    wheel_odometry_calibration_example,
    kinematic_position_velocity_example,
    lidar_plane_fitting_example,
)

print("✅ Successfully imported position_class estimation suite!")


---
## 📦 Module 1: Least Squares Foundations & Where $\mathbf{H}$ and $\mathbf{P}$ Come From

### 1. The Observation Matrix $\mathbf{H}$
In autonomous driving, sensors output measurements $\mathbf{y} \in \mathbb{R}^m$ that relate to latent physical states $\mathbf{x} \in \mathbb{R}^n$ via a function $\mathbf{h}(\mathbf{x})$:
$$\mathbf{y} = \mathbf{h}(\mathbf{x}) + \mathbf{v}, \quad \mathbf{v} \sim \mathcal{N}(\mathbf{0}, \mathbf{R})$$

The matrix $\mathbf{H} \in \mathbb{R}^{m \times n}$ is the **Jacobian matrix** of the measurement function with respect to the parameter vector $\mathbf{x}$:
$$\mathbf{H} = \frac{\partial \mathbf{h}(\mathbf{x})}{\partial \mathbf{x}} = \begin{bmatrix}
\frac{\partial h_1}{\partial x_1} & \cdots & \frac{\partial h_1}{\partial x_n} \\
\vdots & \ddots & \vdots \\
\frac{\partial h_m}{\partial x_1} & \cdots & \frac{\partial h_m}{\partial x_n}
\end{bmatrix}$$

### 2. The Parameter Error Covariance Matrix $\mathbf{P}$
By minimizing the weighted loss $J(\mathbf{x}) = \frac{1}{2}(\mathbf{y} - \mathbf{H}\mathbf{x})^T \mathbf{R}^{-1} (\mathbf{y} - \mathbf{H}\mathbf{x})$, we obtain the **Best Linear Unbiased Estimator (BLUE)**:
$$\hat{\mathbf{x}} = (\mathbf{H}^T \mathbf{R}^{-1} \mathbf{H})^{-1} \mathbf{H}^T \mathbf{R}^{-1} \mathbf{y}$$
$$\mathbf{P} = \operatorname{Cov}(\hat{\mathbf{x}} - \mathbf{x}) = (\mathbf{H}^T \mathbf{R}^{-1} \mathbf{H})^{-1}$$


In [ ]:
# 1. Ohm's Law Parameter Estimation (V = R * I)
I_data = np.array([[0.2, 0.3, 0.4, 0.5, 0.6]]).T
V_data = np.array([[1.23, 1.38, 2.06, 2.47, 3.17]]).T

bls = BatchLeastSquares()
res_ohms = bls.fit(H=I_data, y=V_data)
R_hat = float(res_ohms.x_hat[0, 0])
R_std = float(np.sqrt(res_ohms.covariance[0, 0]))

# 2. Wheel Odometry Calibration (v_gps = r_eff * omega)
r_eff, r_std = wheel_odometry_calibration_example()

print(f"🔹 Estimated Resistor R:        {R_hat:.4f} Ω  (± {3*R_std:.4f} Ω at 3-sigma)")
print(f"🔹 Calibrated Wheel Radius r:   {r_eff:.4f} m  (± {3*r_std:.4f} m at 3-sigma)")

# Interactive Plotly Visualization
fig_ls = make_subplots(rows=1, cols=2, subplot_titles=("Ohm's Law: Voltage vs. Current", "Autonomous Vehicle Wheel Odometry Calibration"))

# Plot 1: Ohm's law
I_dense = np.linspace(0, 0.7, 50)
fig_ls.add_trace(go.Scatter(x=I_data.ravel(), y=V_data.ravel(), mode='markers', marker=dict(size=10, color='red'), name='Multimeter V'), row=1, col=1)
fig_ls.add_trace(go.Scatter(x=I_dense, y=I_dense * R_hat, mode='lines', line=dict(color='blue', dash='solid'), name=f'Fit: R = {R_hat:.2f} Ω'), row=1, col=1)

# Plot 2: Odometry calibration
omega_sim = np.array([10.0, 20.0, 30.0, 40.0, 50.0, 60.0])
v_gps_sim = np.array([3.28, 6.64, 9.87, 13.25, 16.48, 19.82])
omega_dense = np.linspace(0, 70, 50)
fig_ls.add_trace(go.Scatter(x=omega_sim, y=v_gps_sim, mode='markers', marker=dict(size=10, color='darkorange'), name='GPS Speed Pings'), row=1, col=2)
fig_ls.add_trace(go.Scatter(x=omega_dense, y=omega_dense * r_eff, mode='lines', line=dict(color='green', dash='dash'), name=f'Calibrated r_eff = {r_eff:.3f} m'), row=1, col=2)

fig_ls.update_xaxes(title_text="Current I (Amperes)", row=1, col=1)
fig_ls.update_yaxes(title_text="Voltage V (Volts)", row=1, col=1)
fig_ls.update_xaxes(title_text="Wheel Encoder Angular Rate ω (rad/s)", row=1, col=2)
fig_ls.update_yaxes(title_text="Vehicle GPS Speed (m/s)", row=1, col=2)
fig_ls.update_layout(title="Module 1: Batch Least Squares Physical Parameter Calibrations", template="plotly_white", height=450)
fig_ls.show()


---
## 🚗 Module 2: The Discrete Linear Kalman Filter (LKF)

### 1. State-Space Dynamic Model
$$\mathbf{x}_k = \mathbf{F}_{k-1}\mathbf{x}_{k-1} + \mathbf{G}_{k-1}\mathbf{u}_{k-1} + \mathbf{w}_{k-1}, \quad \mathbf{w}_{k-1} \sim \mathcal{N}(\mathbf{0}, \mathbf{Q}_{k-1})$$
$$\mathbf{y}_k = \mathbf{H}_k\mathbf{x}_k + \mathbf{v}_k, \quad \mathbf{v}_k \sim \mathcal{N}(\mathbf{0}, \mathbf{R}_k)$$

### 2. The 5 Core LKF Equations
1. **State Prediction:** $\check{\mathbf{x}}_k = \mathbf{F}_{k-1}\hat{\mathbf{x}}_{k-1} + \mathbf{G}_{k-1}\mathbf{u}_{k-1}$
2. **Covariance Prediction:** $\check{\mathbf{P}}_k = \mathbf{F}_{k-1}\hat{\mathbf{P}}_{k-1}\mathbf{F}_{k-1}^T + \mathbf{Q}_{k-1}$
3. **Kalman Gain:** $\mathbf{K}_k = \check{\mathbf{P}}_k \mathbf{H}_k^T (\mathbf{H}_k \check{\mathbf{P}}_k \mathbf{H}_k^T + \mathbf{R}_k)^{-1}$
4. **State Correction:** $\hat{\mathbf{x}}_k = \check{\mathbf{x}}_k + \mathbf{K}_k (\mathbf{y}_k - \mathbf{H}_k \check{\mathbf{x}}_k)$
5. **Covariance Correction:** $\hat{\mathbf{P}}_k = (\mathbf{I} - \mathbf{K}_k \mathbf{H}_k)\check{\mathbf{P}}_k$


In [ ]:
# Simulate a 2D vehicle trajectory with Constant Velocity tracking
np.random.seed(42)
dt = 0.1
total_time = 30.0
steps = int(total_time / dt)
time_arr = np.linspace(0, total_time, steps)

# True kinematics: vehicle accelerates then drives straight and turns
true_x = np.zeros(steps)
true_y = np.zeros(steps)
true_vx = np.zeros(steps)
true_vy = np.zeros(steps)

vx_curr = 12.0 # 12 m/s (~43 km/h)
vy_curr = 0.0

for k in range(1, steps):
    if 5.0 <= time_arr[k] <= 15.0:
        vy_curr = 4.0 * np.sin(2 * np.pi * (time_arr[k] - 5.0) / 10.0)
    true_vx[k] = vx_curr
    true_vy[k] = vy_curr
    true_x[k] = true_x[k-1] + true_vx[k-1] * dt
    true_y[k] = true_y[k-1] + true_vy[k-1] * dt

# Generate noisy GPS position measurements (sigma = 3.0 m)
sigma_gps = 3.0
meas_x = true_x + np.random.normal(0, sigma_gps, steps)
meas_y = true_y + np.random.normal(0, sigma_gps, steps)

# Instantiate LKF Tracker
tracker = create_2d_constant_velocity_tracker(dt=dt, sigma_pos_gps=sigma_gps, sigma_acc_process=0.5)

# Run Filter Loop
est_x = np.zeros(steps)
est_y = np.zeros(steps)
est_vx = np.zeros(steps)
est_vy = np.zeros(steps)
cov_x = np.zeros(steps)

for k in range(steps):
    tracker.predict()
    state = tracker.update(np.array([meas_x[k], meas_y[k]]))
    est_x[k] = state.x[0, 0]
    est_y[k] = state.x[1, 0]
    est_vx[k] = state.x[2, 0]
    est_vy[k] = state.x[3, 0]
    cov_x[k] = state.P[0, 0]

rmse_gps = np.sqrt(np.mean((meas_x - true_x)**2 + (meas_y - true_y)**2))
rmse_lkf = np.sqrt(np.mean((est_x - true_x)**2 + (est_y - true_y)**2))

print(f"📊 2D Tracking Results:")
print(f"   Raw GPS Position RMSE: {rmse_gps:.3f} m")
print(f"   LKF Filtered RMSE:     {rmse_lkf:.3f} m  (Error reduced by {100*(1 - rmse_lkf/rmse_gps):.1f}%)")

# Plot Trajectories
fig_lkf = go.Figure()
fig_lkf.add_trace(go.Scatter(x=true_x, y=true_y, mode='lines', line=dict(color='black', width=3), name='Ground Truth Path'))
fig_lkf.add_trace(go.Scatter(x=meas_x[::5], y=meas_y[::5], mode='markers', marker=dict(size=5, color='red', opacity=0.6), name='Raw GPS Pings'))
fig_lkf.add_trace(go.Scatter(x=est_x, y=est_y, mode='lines', line=dict(color='blue', width=2), name='LKF Estimated Path'))

fig_lkf.update_layout(
    title=f"Module 2: 2D Vehicle Tracking via Linear Kalman Filter (RMSE: {rmse_lkf:.2f} m vs. GPS: {rmse_gps:.2f} m)",
    xaxis_title="East Position (m)",
    yaxis_title="North Position (m)",
    template="plotly_white",
    height=500
)
fig_lkf.show()


---
## 🎯 Module 3: Extended Kalman Filter (EKF) with Analytical Jacobians

When models are nonlinear, we compute the four Jacobians evaluated at the operating points:
$$\mathbf{F}_{k-1} = \left. \frac{\partial \mathbf{f}}{\partial \mathbf{x}} \right|_{\hat{\mathbf{x}}_{k-1}, \mathbf{u}_{k-1}, \mathbf{0}}, \quad \mathbf{L}_{k-1} = \left. \frac{\partial \mathbf{f}}{\partial \mathbf{w}} \right|_{\hat{\mathbf{x}}_{k-1}, \mathbf{u}_{k-1}, \mathbf{0}}$$
$$\mathbf{H}_k = \left. \frac{\partial \mathbf{h}}{\partial \mathbf{x}} \right|_{\check{\mathbf{x}}_k, \mathbf{0}}, \quad \mathbf{M}_k = \left. \frac{\partial \mathbf{h}}{\partial \mathbf{v}} \right|_{\check{\mathbf{x}}_k, \mathbf{0}}$$

### Benchmark: Optical Landmark Bearing Estimation (`The Nonlinear Kalman Filter/Excersice.ipynb`)
* **State:** $\mathbf{x} = [p, v]^T$
* **Bearing observation:** $y_k = \arctan\left(\frac{S}{D - p_k}\right) + v_k$
* **Analytical Measurement Jacobian:** $\mathbf{H}_k = \begin{bmatrix} \frac{S}{(D - \check{p}_k)^2 + S^2} & 0 \end{bmatrix}$


In [ ]:
# Instantiate Landmark Bearing EKF & UKF
dt_nl = 0.5
S_val = 20.0
D_val = 40.0
u_cmd = -2.0 # -2 m/s^2 deceleration

ekf_landmark = LandmarkBearingEKF(dt=dt_nl, S=S_val, D=D_val)
ukf_landmark = LandmarkBearingUKF(dt=dt_nl, S=S_val, D=D_val)

# Perform predict + update step with sensor reading y = pi/6 (30 deg)
y_reading = np.pi / 6.0

x_ekf, P_ekf = ekf_landmark.step(u=u_cmd, y=y_reading)
x_ukf, P_ukf = ukf_landmark.step(u=u_cmd, y=y_reading)

df_nl = pl.DataFrame({
    "Estimator": ["EKF (Analytical Jacobians)", "UKF (Sigma Points / Zero Jacobians)"],
    "Estimated Position (m)": [round(float(x_ekf[0, 0]), 4), round(float(x_ukf[0, 0]), 4)],
    "Estimated Velocity (m/s)": [round(float(x_ekf[1, 0]), 4), round(float(x_ukf[1, 0]), 4)],
    "Pos Variance (m²)": [round(float(P_ekf[0, 0]), 6), round(float(P_ukf[0, 0]), 6)],
    "Vel Variance ((m/s)²)": [round(float(P_ekf[1, 1]), 6), round(float(P_ukf[1, 1]), 6)],
})

print("🏆 Landmark Bearing Step Comparison (Exercise.ipynb):")
print(df_nl)


---
## 🔬 Module 4: 2D Radar Target Tracking (Range & Bearing) & UKF Benchmark

An ego-vehicle tracks an obstacle using a 2D Radar sensor that outputs polar measurements:
$$\mathbf{y}_k = \begin{bmatrix} r \\ \phi \end{bmatrix} = \begin{bmatrix} \sqrt{x^2 + y^2} \\ \operatorname{atan2}(y, x) \end{bmatrix} + \mathbf{v}_k$$


In [ ]:
# Simulate a moving obstacle tracked by 2D Polar Radar
np.random.seed(101)
radar_dt = 0.1
radar_steps = 150
radar_time = np.linspace(0, radar_steps * radar_dt, radar_steps)

# Obstacle ground truth trajectory
tgt_x = 10.0 + 8.0 * np.cos(0.2 * radar_time)
tgt_y = 15.0 + 8.0 * np.sin(0.2 * radar_time)
tgt_vx = -1.6 * np.sin(0.2 * radar_time)
tgt_vy = 1.6 * np.cos(0.2 * radar_time)

# Simulated polar Radar observations
sigma_r = 0.4     # 0.4 m range noise
sigma_phi = 0.025 # 1.4 degrees azimuth noise

meas_r = np.sqrt(tgt_x**2 + tgt_y**2) + np.random.normal(0, sigma_r, radar_steps)
meas_phi = np.arctan2(tgt_y, tgt_x) + np.random.normal(0, sigma_phi, radar_steps)

# Run EKF Radar Tracker
radar_ekf = Radar2DTargetTrackerEKF(dt=radar_dt, sigma_range=sigma_r, sigma_bearing=sigma_phi, x0=np.array([10.0, 15.0, 0.0, 1.6]))

ekf_traj_x = np.zeros(radar_steps)
ekf_traj_y = np.zeros(radar_steps)

for k in range(radar_steps):
    radar_ekf.ekf.predict()
    x_hat, _ = radar_ekf.ekf.update(np.array([meas_r[k], meas_phi[k]]))
    ekf_traj_x[k] = x_hat[0, 0]
    ekf_traj_y[k] = x_hat[1, 0]

rmse_radar_ekf = np.sqrt(np.mean((ekf_traj_x - tgt_x)**2 + (ekf_traj_y - tgt_y)**2))
print(f"🎯 2D Polar Radar Tracking RMSE: {rmse_radar_ekf:.3f} m")

# Plot Radar Tracking
fig_radar = go.Figure()
fig_radar.add_trace(go.Scatter(x=[0], y=[0], mode='markers', marker=dict(size=14, symbol='square', color='purple'), name='Ego Radar Sensor (0,0)'))
fig_radar.add_trace(go.Scatter(x=tgt_x, y=tgt_y, mode='lines', line=dict(color='black', width=3), name='True Obstacle Path'))

# Plot noisy cartesian projections from radar
raw_cart_x = meas_r * np.cos(meas_phi)
raw_cart_y = meas_r * np.sin(meas_phi)
fig_radar.add_trace(go.Scatter(x=raw_cart_x[::3], y=raw_cart_y[::3], mode='markers', marker=dict(size=5, color='orange', opacity=0.7), name='Raw Radar Detections'))
fig_radar.add_trace(go.Scatter(x=ekf_traj_x, y=ekf_traj_y, mode='lines', line=dict(color='green', width=2), name='EKF Tracked Path'))

fig_radar.update_layout(
    title=f"Module 4: 2D Radar Obstacle Tracking with Polar Range/Bearing Measurements (RMSE: {rmse_radar_ekf:.2f} m)",
    xaxis_title="Lateral Position X (m)",
    yaxis_title="Longitudinal Position Y (m)",
    template="plotly_white",
    height=500
)
fig_radar.show()


---
## 🏁 Summary & Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────────────────┐
│                   STATE ESTIMATION SUMMARY & TAXONOMY                            │
├───────────────────┬───────────────────────────────┬──────────────────────────────┤
│ Method            │ Key Strengths                 │ Limitations                  │
├───────────────────┼───────────────────────────────┼──────────────────────────────┤
│ Batch Least Sq.   │ Global parameter optimal      │ Non-recursive, high memory   │
│ Linear Kalman (LKF)│ Real-time, optimal for linear │ Fails on nonlinear systems   │
│ Extended KF (EKF) │ Industry standard, fast       │ Jacobians prone to error     │
│ Unscented KF (UKF)│ Zero Jacobians, 3rd order acc │ 2n+1 function evaluations    │
└───────────────────┴───────────────────────────────┴──────────────────────────────┘
```

### 📚 Next Steps:
* Explore the codebase in [`position_class/src/position_class/`](file:///Users/yasuomaidana/Projects/classes/Self-Driving-Cars/State%20Estimation%20and%20Localization%20for%20Self-Driving%20Cars/position_class/src/position_class/).
* Review theoretical derivations in [`Day_01_Foundations_of_State_Estimation`](file:///Users/yasuomaidana/Projects/classes/Self-Driving-Cars/State%20Estimation%20and%20Localization%20for%20Self-Driving%20Cars/Day_01_Foundations_of_State_Estimation) and [`Day_02_Nonlinear_Estimation_and_Object_Tracking`](file:///Users/yasuomaidana/Projects/classes/Self-Driving-Cars/State%20Estimation%20and%20Localization%20for%20Self-Driving%20Cars/Day_02_Nonlinear_Estimation_and_Object_Tracking).
